# IOAI — 2025 Gaite Contest Synthetic Speech Detector — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "GAITE-Contest/Synthetic_Speech_Detector"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

In [ ]:
!pip install  dataset

In [ ]:
import os
import zipfile
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import resnet34, ResNet34_Weights

#from dataset.spectrogram_dataset import SpectrogramDataset
import os
import torch
from torch.utils.data import Dataset

In [ ]:
class SpectrogramDataset(Dataset):
    """
    Load spectrogram data from preprocessed .pt files.

    For training_set/, assumes:
        dataset/training_set/
            bonafide/
            spoof/

    For validation_set/ and testing_set/, assumes:
        dataset/validation_set/    (all .pt files in this folder, no subfolders)
        dataset/testing_set/       (all .pt files in this folder, no subfolders)

    No label will be provided for val/test sets to prevent label leakage.
    """

    def __init__(self, directory):
        self.samples = []

        if "training" in directory:
            label_map = {"bonafide": 0, "spoof": 1}
            for label_name, label in label_map.items():
                label_dir = os.path.join(directory, label_name)
                if not os.path.isdir(label_dir):
                    continue
                for fname in os.listdir(label_dir):
                    if fname.endswith(".pt"):
                        self.samples.append(
                            {"path": os.path.join(label_dir, fname), "label": label}
                        )
        else:
            for fname in sorted(os.listdir(directory)):
                if fname.endswith(".pt"):
                    self.samples.append({"path": os.path.join(directory, fname)})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        spec = torch.load(item["path"])
        out = {"spectrogram": spec}
        if "label" in item:
            out["label"] = torch.tensor(item["label"], dtype=torch.long)
        return out


In [ ]:
class AudioNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet34(weights=ResNet34_Weights.DEFAULT)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight = nn.Parameter(
                base.conv1.weight.mean(dim=1, keepdim=True)
            )
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(base.fc.in_features, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [ ]:
def train_one_epoch(model, train_loader, val_loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="Train"):
        x = batch["spectrogram"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss/= len(train_loader)
    print(f"Train Loss: {train_loss:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val Split"):
            x = batch["spectrogram"].to(device)
            y = batch["label"].to(device)
            output = model(x)
            loss = criterion(output, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print(f"Val Split Loss: {val_loss:.4f}")

In [ ]:
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            x = batch["spectrogram"].to(device)
            output = model(x)
            pred = torch.argmax(output, dim=1)
            preds.extend(pred.cpu().numpy())
    return preds

In [ ]:
def save_submission_csv(preds, save_name):
    df = pd.DataFrame(preds)
    df.to_csv(save_name, index=False, header=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
full_train_set = SpectrogramDataset("training_set/")

val_size = int(0.2 * len(full_train_set))
train_size = len(full_train_set) - val_size

train_set, val_split_set = random_split(full_train_set, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_split_loader = DataLoader(val_split_set, batch_size=32)

for _ in range(2):
    train_one_epoch(
        model, train_loader, val_split_loader, criterion, optimizer, device
    )

In [ ]:
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH")+"/" 
else:
    DATA_PATH = "test_v3"
    
val_set = SpectrogramDataset(DATA_PATH + "/validation_set")
test_set = SpectrogramDataset(DATA_PATH + "/testing_set")

val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

val_preds = predict(model, val_loader, device)
test_preds = predict(model, test_loader, device)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)